# UVIF-Guided Resource-Aware Secure Boundary Estimation for BB84-Inspired Finite-Key Quantum Communications

This upgraded notebook supports the manuscript **“UVIF-Guided Resource-Aware Secure Boundary Estimation for BB84-Inspired Finite-Key Quantum Communications.”**

It extends a conventional finite-key BB84-inspired boundary-estimation workflow into a UVIF-guided operational security analysis. The notebook generates:

- conventional QBER-threshold and finite-key margin baselines,
- UVIF operational free-energy and equilibrium-force diagnostics,
- resource-aware secure-to-insecure boundary estimates,
- bootstrap uncertainty intervals with a runtime-conscious default configuration,
- shot-count and phase-margin robustness tests,
- temporal trajectory analysis with early-warning indicators,
- publication-ready tables, figures, and an `outputs_summary.txt` file.

All generated outputs are saved under a run-specific folder in Google Drive when executed in Colab, or under a local `Outputs/` directory otherwise.


In [ ]:
import json
import os
import platform
import random
import sys
import warnings
from datetime import datetime
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

IN_COLAB = False
try:
    import google.colab
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    DRIVE_ROOT = "/content/drive"
    MOUNT_POINT = f"{DRIVE_ROOT}/MyDrive"
    try:
        drive.mount(DRIVE_ROOT, force_remount=False)
    except Exception as exc:
        print(f"[WARN] Google Drive mount issue: {exc}")
    BASE_ROOT = Path(MOUNT_POINT) / "Outputs"
else:
    BASE_ROOT = Path("Outputs")

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
PROJECT_NAME = "UVIF_BB84_FiniteKey_Boundary_Dynamics"
PROJECT_ROOT = BASE_ROOT / PROJECT_NAME / f"run_{RUN_ID}"

FIG_DIR = PROJECT_ROOT / "figures"
TAB_DIR = PROJECT_ROOT / "tables"
MODEL_DIR = PROJECT_ROOT / "models"
OUT_DIR = PROJECT_ROOT / "outputs"
LOG_DIR = PROJECT_ROOT / "logs"
PYTHON_DIR = PROJECT_ROOT / "python"

for directory in [FIG_DIR, TAB_DIR, MODEL_DIR, OUT_DIR, LOG_DIR, PYTHON_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

def log(message):
    print(message)
    with open(LOG_DIR / "run_log.txt", "a", encoding="utf-8") as f:
        f.write(str(message) + "\n")

manifest = {
    "project_name": PROJECT_NAME,
    "run_id": RUN_ID,
    "seed": SEED,
    "in_colab": IN_COLAB,
    "python_version": sys.version.split()[0],
    "platform": platform.platform(),
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "matplotlib_version": matplotlib.__version__,
    "project_root": str(PROJECT_ROOT),
    "figures": str(FIG_DIR),
    "tables": str(TAB_DIR),
    "outputs": str(OUT_DIR),
}

with open(OUT_DIR / "run_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

log(f"[RUN] {RUN_ID}")
log(f"[ROOT] {PROJECT_ROOT}")
log(f"[ENV] Colab={IN_COLAB}, Python={manifest['python_version']}, NumPy={np.__version__}, Pandas={pd.__version__}")


In [ ]:
# ----------------------------
# Study configuration
# ----------------------------
TIME_HORIZON = 120
T = np.arange(TIME_HORIZON, dtype=float)
EPS = 1e-12

# Conventional BB84-inspired descriptors
QBER_SECURITY_THRESHOLD = 0.11
PHASE_MARGIN = 0.02

# Operational controls
ETA_GRID = np.linspace(0.01, 0.22, 61)                    # disturbance
ATTACK_GRID = np.array([0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40])
DRIFT_GRID = np.array([0.005, 0.010, 0.020, 0.030])

# Resource controls
SHOT_COUNTS = np.array([2_000, 5_000, 10_000, 20_000])
DEFAULT_SHOTS = 10_000
N_REPEATS = 100
BOOTSTRAPS = 160

# UVIF weights.
# The notation follows a resource-aware variational interpretation:
# low free energy corresponds to a more stable and operationally secure regime.
UVIF_WEIGHTS = {
    "risk": 1.00,            # QBER/attack-driven operational risk
    "uncertainty": 0.75,     # finite-key uncertainty and predictive entropy
    "complexity": 0.45,      # implementation drift and control complexity
    "resource": 0.55,        # resource pressure induced by low shot count/effective detections
    "leakage": 0.65,         # error-correction/privacy-amplification leakage burden
    "stability_reward": 0.80 # reward for finite-key secrecy margin
}

UVIF_BOUNDARY_LEVEL = 0.50

SCENARIOS = [
    {"name": "secure_reference", "eta": 0.035, "attack": 0.08, "drift": 0.008, "shots": DEFAULT_SHOTS},
    {"name": "near_boundary", "eta": 0.075, "attack": 0.16, "drift": 0.015, "shots": DEFAULT_SHOTS},
    {"name": "resource_limited_transition", "eta": 0.080, "attack": 0.18, "drift": 0.015, "shots": 2_000},
    {"name": "elevated_pressure", "eta": 0.105, "attack": 0.24, "drift": 0.020, "shots": DEFAULT_SHOTS},
    {"name": "insecure_reference", "eta": 0.145, "attack": 0.32, "drift": 0.030, "shots": DEFAULT_SHOTS},
]

config = {
    "time_horizon": TIME_HORIZON,
    "qber_security_threshold": QBER_SECURITY_THRESHOLD,
    "phase_margin": PHASE_MARGIN,
    "eta_min": float(ETA_GRID.min()),
    "eta_max": float(ETA_GRID.max()),
    "eta_points": int(len(ETA_GRID)),
    "attack_grid": ATTACK_GRID.tolist(),
    "drift_grid": DRIFT_GRID.tolist(),
    "shot_counts": SHOT_COUNTS.tolist(),
    "default_shots": int(DEFAULT_SHOTS),
    "n_repeats": int(N_REPEATS),
    "bootstraps": int(BOOTSTRAPS),
    "uvif_weights": UVIF_WEIGHTS,
    "uvif_boundary_level": UVIF_BOUNDARY_LEVEL,
    "scenarios": SCENARIOS,
}

with open(OUT_DIR / "study_configuration.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

log("[CONFIG] Configuration saved.")
log(f"[CONFIG] eta grid={len(ETA_GRID)}, attack levels={len(ATTACK_GRID)}, drift levels={len(DRIFT_GRID)}, repeats={N_REPEATS}")


In [ ]:
# ----------------------------
# Finite-key BB84-inspired model and UVIF descriptors
# ----------------------------
def clip01(x):
    return np.clip(x, 0.0, 1.0)

def binary_entropy(p):
    p = np.clip(np.asarray(p, dtype=float), EPS, 1.0 - EPS)
    return -(p * np.log2(p) + (1.0 - p) * np.log2(1.0 - p))

def deterministic_qber_mean(eta, attack, drift, t, time_horizon=TIME_HORIZON):
    seasonal = 0.006 * np.sin(2.0 * np.pi * t / max(time_horizon, 1))
    slow_trend = 0.010 * (t / max(time_horizon - 1, 1))
    qber = eta + 0.055 * attack + 0.90 * drift + seasonal + slow_trend
    return float(clip01(qber))

def effective_detection_count(shots, attack, drift):
    retention = 1.0 - 0.18 * attack - 1.10 * drift
    retention = float(np.clip(retention, 0.35, 1.0))
    return max(50, int(round(shots * retention)))

def sample_observed_qber(qber_mean, shots, attack, drift, rng):
    n_eff = effective_detection_count(shots, attack=attack, drift=drift)
    errors = rng.binomial(n=n_eff, p=float(np.clip(qber_mean, EPS, 1.0 - EPS)))
    return float(errors / n_eff), n_eff

def finite_key_penalty(n_eff):
    return float(2.6 / np.sqrt(max(n_eff, 1)))

def secrecy_margin(qber_obs, n_eff, drift):
    # BB84-inspired finite-key margin using an asymptotic entropy term plus finite-key and drift penalties.
    margin = 1.0 - 2.0 * binary_entropy(np.minimum(qber_obs, 0.499999)) - finite_key_penalty(n_eff) - 1.35 * drift
    return float(margin)

def leakage_burden(qber_obs, attack, drift):
    return float(0.65 * attack + 0.75 * qber_obs + 0.90 * drift)

def qber_threshold_margin(qber_obs):
    return float(QBER_SECURITY_THRESHOLD - qber_obs)

def order_parameter_from_margin(margin):
    return float(margin / max(abs(QBER_SECURITY_THRESHOLD), EPS))

def classify_phase(psi, margin=PHASE_MARGIN):
    if psi > margin:
        return "secure"
    if psi < -margin:
        return "insecure"
    return "transition"

def finite_key_uncertainty(qber_obs, n_eff):
    # Normal-approximation uncertainty plus entropy uncertainty proxy.
    sampling = np.sqrt(max(qber_obs * (1.0 - qber_obs), EPS) / max(n_eff, 1))
    entropy_proxy = binary_entropy(np.clip(qber_obs, EPS, 1.0 - EPS))
    return float(sampling + 0.05 * entropy_proxy)

def resource_pressure(n_eff, reference_shots=DEFAULT_SHOTS):
    return float(np.clip(np.sqrt(reference_shots / max(n_eff, 1)) - 1.0, 0.0, 2.0))

def uvif_force_terms(qber_obs, qber_mean, n_eff, attack, drift, margin):
    risk_force = float(np.clip(qber_obs / QBER_SECURITY_THRESHOLD, 0.0, 3.0))
    uncertainty_force = finite_key_uncertainty(qber_obs, n_eff)
    complexity_force = float(0.70 * drift + 0.30 * abs(qber_obs - qber_mean))
    resource_force = resource_pressure(n_eff)
    leakage_force = leakage_burden(qber_obs, attack, drift)
    stability_reward = float(np.tanh(max(margin, -1.0)))
    return {
        "risk_force": risk_force,
        "uncertainty_force": uncertainty_force,
        "complexity_force": complexity_force,
        "resource_force": resource_force,
        "leakage_force": leakage_force,
        "stability_reward": stability_reward,
    }

def uvif_operational_free_energy(force_terms, weights=UVIF_WEIGHTS):
    energy = (
        weights["risk"] * force_terms["risk_force"]
        + weights["uncertainty"] * force_terms["uncertainty_force"]
        + weights["complexity"] * force_terms["complexity_force"]
        + weights["resource"] * force_terms["resource_force"]
        + weights["leakage"] * force_terms["leakage_force"]
        - weights["stability_reward"] * force_terms["stability_reward"]
    )
    return float(energy)

def uvif_stability_score(free_energy):
    # Higher score means safer/more stable. The level 0.5 corresponds to free_energy = 0.
    return float(1.0 / (1.0 + np.exp(free_energy)))

def classify_uvif_regime(uvif_score):
    if uvif_score >= 0.62:
        return "uvif_stable_secure"
    if uvif_score <= 0.38:
        return "uvif_unstable_insecure"
    return "uvif_transition"

def susceptibility(values, controls):
    return np.gradient(np.asarray(values, dtype=float), np.asarray(controls, dtype=float))

def first_transition_time(series, threshold=0.0, direction="below"):
    arr = np.asarray(series, dtype=float)
    if direction == "below":
        idx = np.where(arr <= threshold)[0]
    else:
        idx = np.where(arr >= threshold)[0]
    return int(idx[0]) if len(idx) else -1

# Smoke test
rng_smoke = np.random.default_rng(SEED)
qber_mean_smoke = deterministic_qber_mean(eta=0.06, attack=0.10, drift=0.01, t=15)
qber_obs_smoke, n_eff_smoke = sample_observed_qber(qber_mean_smoke, DEFAULT_SHOTS, 0.10, 0.01, rng_smoke)
margin_smoke = secrecy_margin(qber_obs_smoke, n_eff_smoke, 0.01)
forces_smoke = uvif_force_terms(qber_obs_smoke, qber_mean_smoke, n_eff_smoke, 0.10, 0.01, margin_smoke)
energy_smoke = uvif_operational_free_energy(forces_smoke)
score_smoke = uvif_stability_score(energy_smoke)

log(f"[SMOKE] qber_mean={qber_mean_smoke:.6f}, qber_obs={qber_obs_smoke:.6f}, n_eff={n_eff_smoke}")
log(f"[SMOKE] finite_key_margin={margin_smoke:.6f}, psi={order_parameter_from_margin(margin_smoke):.6f}")
log(f"[SMOKE] uvif_energy={energy_smoke:.6f}, uvif_score={score_smoke:.6f}, uvif_regime={classify_uvif_regime(score_smoke)}")


In [ ]:
# ----------------------------
# Stochastic trajectory simulator
# ----------------------------
def simulate_trajectory(eta, attack, drift, shots=DEFAULT_SHOTS, time_horizon=TIME_HORIZON, seed=None):
    rng = np.random.default_rng(SEED if seed is None else seed)
    rows = []

    for t in range(time_horizon):
        qber_mean = deterministic_qber_mean(eta, attack, drift, t, time_horizon)
        qber_obs, n_eff = sample_observed_qber(qber_mean, shots, attack, drift, rng)
        margin = secrecy_margin(qber_obs, n_eff, drift)
        psi = order_parameter_from_margin(margin)
        qber_margin = qber_threshold_margin(qber_obs)
        forces = uvif_force_terms(qber_obs, qber_mean, n_eff, attack, drift, margin)
        uvif_energy = uvif_operational_free_energy(forces)
        uvif_score = uvif_stability_score(uvif_energy)

        rows.append({
            "t": int(t),
            "eta": float(eta),
            "attack": float(attack),
            "drift": float(drift),
            "shots": int(shots),
            "qber_mean": float(qber_mean),
            "qber_obs": float(qber_obs),
            "n_eff": int(n_eff),
            "qber_threshold_margin": float(qber_margin),
            "finite_key_margin": float(margin),
            "psi": float(psi),
            "conventional_phase": classify_phase(psi),
            "uvif_energy": float(uvif_energy),
            "uvif_score": float(uvif_score),
            "uvif_regime": classify_uvif_regime(uvif_score),
            **forces,
        })

    df = pd.DataFrame(rows)
    df["d_qber_dt"] = np.gradient(df["qber_obs"].values, df["t"].values)
    df["d_psi_dt"] = np.gradient(df["psi"].values, df["t"].values)
    df["d_uvif_score_dt"] = np.gradient(df["uvif_score"].values, df["t"].values)
    df["early_warning_index"] = (
        0.45 * np.clip(-df["d_psi_dt"], 0, None)
        + 0.35 * np.clip(-df["d_uvif_score_dt"], 0, None)
        + 0.20 * np.clip(df["d_qber_dt"], 0, None)
    )
    return df

def simulate_endpoint_replicates(eta, attack, drift, shots=DEFAULT_SHOTS, n_repeats=N_REPEATS, rng=None):
    if rng is None:
        rng = np.random.default_rng(SEED)

    rows = []
    for repeat in range(n_repeats):
        qber_mean = deterministic_qber_mean(eta, attack, drift, TIME_HORIZON - 1, TIME_HORIZON)
        qber_obs, n_eff = sample_observed_qber(qber_mean, shots, attack, drift, rng)
        margin = secrecy_margin(qber_obs, n_eff, drift)
        psi = order_parameter_from_margin(margin)
        qber_margin = qber_threshold_margin(qber_obs)
        forces = uvif_force_terms(qber_obs, qber_mean, n_eff, attack, drift, margin)
        uvif_energy = uvif_operational_free_energy(forces)
        uvif_score = uvif_stability_score(uvif_energy)

        rows.append({
            "eta": float(eta),
            "attack": float(attack),
            "drift": float(drift),
            "shots": int(shots),
            "repeat": int(repeat),
            "qber_mean": float(qber_mean),
            "qber_obs": float(qber_obs),
            "n_eff": int(n_eff),
            "qber_threshold_margin": float(qber_margin),
            "finite_key_margin": float(margin),
            "psi": float(psi),
            "conventional_phase": classify_phase(psi),
            "uvif_energy": float(uvif_energy),
            "uvif_score": float(uvif_score),
            "uvif_regime": classify_uvif_regime(uvif_score),
            **forces,
        })
    return pd.DataFrame(rows)

test_traj = simulate_trajectory(eta=0.075, attack=0.16, drift=0.015, shots=DEFAULT_SHOTS, seed=SEED + 11)
log(f"[TRAJECTORY TEST] rows={len(test_traj)}, final_uvif_score={test_traj['uvif_score'].iloc[-1]:.4f}")


In [ ]:
# ----------------------------
# Endpoint sweep: conventional finite-key and UVIF-guided state summaries
# ----------------------------
endpoint_frames = []
summary_rows = []
main_rng = np.random.default_rng(SEED + 1000)

total_jobs = len(DRIFT_GRID) * len(ATTACK_GRID) * len(ETA_GRID)
job_id = 0

for drift in DRIFT_GRID:
    for attack in ATTACK_GRID:
        for eta in ETA_GRID:
            job_id += 1
            df = simulate_endpoint_replicates(
                eta=float(eta),
                attack=float(attack),
                drift=float(drift),
                shots=DEFAULT_SHOTS,
                n_repeats=N_REPEATS,
                rng=main_rng,
            )
            endpoint_frames.append(df)

            summary_rows.append({
                "eta": float(eta),
                "attack": float(attack),
                "drift": float(drift),
                "shots": int(DEFAULT_SHOTS),
                "qber_obs_mean": float(df["qber_obs"].mean()),
                "qber_obs_sd": float(df["qber_obs"].std()),
                "n_eff_mean": float(df["n_eff"].mean()),
                "finite_key_margin_mean": float(df["finite_key_margin"].mean()),
                "finite_key_margin_sd": float(df["finite_key_margin"].std()),
                "psi_mean": float(df["psi"].mean()),
                "p_conventional_secure": float((df["conventional_phase"] == "secure").mean()),
                "p_conventional_transition": float((df["conventional_phase"] == "transition").mean()),
                "p_conventional_insecure": float((df["conventional_phase"] == "insecure").mean()),
                "uvif_energy_mean": float(df["uvif_energy"].mean()),
                "uvif_energy_sd": float(df["uvif_energy"].std()),
                "uvif_score_mean": float(df["uvif_score"].mean()),
                "p_uvif_stable": float((df["uvif_regime"] == "uvif_stable_secure").mean()),
                "p_uvif_transition": float((df["uvif_regime"] == "uvif_transition").mean()),
                "p_uvif_unstable": float((df["uvif_regime"] == "uvif_unstable_insecure").mean()),
                "risk_force_mean": float(df["risk_force"].mean()),
                "uncertainty_force_mean": float(df["uncertainty_force"].mean()),
                "complexity_force_mean": float(df["complexity_force"].mean()),
                "resource_force_mean": float(df["resource_force"].mean()),
                "leakage_force_mean": float(df["leakage_force"].mean()),
                "stability_reward_mean": float(df["stability_reward"].mean()),
            })

endpoint_df = pd.concat(endpoint_frames, ignore_index=True)
endpoint_summary_df = pd.DataFrame(summary_rows).sort_values(["drift", "attack", "eta"]).reset_index(drop=True)

endpoint_df.to_csv(TAB_DIR / "endpoint_replicates_full.csv", index=False)
endpoint_summary_df.to_csv(TAB_DIR / "endpoint_summary_uvif.csv", index=False)

log(f"[ENDPOINT] replicate rows={len(endpoint_df):,}")
log(f"[ENDPOINT] summary rows={len(endpoint_summary_df):,}")
log(f"[TABLE] {TAB_DIR / 'endpoint_summary_uvif.csv'}")
display(endpoint_summary_df.head(10))


In [ ]:
# ----------------------------
# Boundary estimation utilities
# ----------------------------
def interpolate_crossing(x, y, target):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float) - float(target)

    exact = np.where(np.isclose(y, 0.0, atol=1e-12))[0]
    if len(exact):
        return float(x[int(exact[0])])

    crossing_idx = np.where(y[:-1] * y[1:] < 0.0)[0]
    if len(crossing_idx) == 0:
        return np.nan

    i = int(crossing_idx[0])
    x0, x1 = x[i], x[i + 1]
    y0, y1 = y[i], y[i + 1]
    if abs(y1 - y0) < EPS:
        return float(0.5 * (x0 + x1))
    return float(x0 + (0.0 - y0) * (x1 - x0) / (y1 - y0))

def bootstrap_probability_boundary(df, eta_values, state_column, positive_label, target=0.5, n_boot=BOOTSTRAPS, seed=SEED):
    rng = np.random.default_rng(seed)
    df = df.copy()
    df["positive"] = (df[state_column] == positive_label).astype(float)

    boundary_samples = []
    grouped = {float(eta): group["positive"].values for eta, group in df.groupby("eta")}
    eta_values = np.asarray(eta_values, dtype=float)

    for _ in range(n_boot):
        probs = []
        for eta in eta_values:
            values = grouped.get(float(eta))
            if values is None or len(values) == 0:
                probs.append(np.nan)
            else:
                sample = rng.choice(values, size=len(values), replace=True)
                probs.append(float(sample.mean()))
        if np.all(np.isnan(probs)):
            continue
        boundary = interpolate_crossing(eta_values, np.asarray(probs), target=target)
        if not np.isnan(boundary):
            boundary_samples.append(boundary)

    if len(boundary_samples) == 0:
        return np.nan, np.nan, np.nan, 0

    lo, med, hi = np.quantile(boundary_samples, [0.025, 0.50, 0.975])
    return float(lo), float(med), float(hi), int(len(boundary_samples))

def estimate_boundaries(endpoint_df, endpoint_summary_df, shots=DEFAULT_SHOTS):
    boundary_rows = []

    for drift in DRIFT_GRID:
        for attack in ATTACK_GRID:
            sub_sum = endpoint_summary_df[
                (endpoint_summary_df["drift"] == float(drift)) &
                (endpoint_summary_df["attack"] == float(attack)) &
                (endpoint_summary_df["shots"] == int(shots))
            ].sort_values("eta")

            sub_rep = endpoint_df[
                (endpoint_df["drift"] == float(drift)) &
                (endpoint_df["attack"] == float(attack)) &
                (endpoint_df["shots"] == int(shots))
            ].copy()

            eta_values = sub_sum["eta"].to_numpy(dtype=float)

            eta_c_qber = interpolate_crossing(
                eta_values,
                sub_sum["qber_obs_mean"].to_numpy(dtype=float),
                target=QBER_SECURITY_THRESHOLD,
            )
            eta_c_margin = interpolate_crossing(
                eta_values,
                sub_sum["finite_key_margin_mean"].to_numpy(dtype=float),
                target=0.0,
            )
            eta_c_conv_p50 = interpolate_crossing(
                eta_values,
                sub_sum["p_conventional_secure"].to_numpy(dtype=float),
                target=0.5,
            )
            eta_c_uvif_score = interpolate_crossing(
                eta_values,
                sub_sum["uvif_score_mean"].to_numpy(dtype=float),
                target=UVIF_BOUNDARY_LEVEL,
            )
            eta_c_uvif_p50 = interpolate_crossing(
                eta_values,
                sub_sum["p_uvif_stable"].to_numpy(dtype=float),
                target=0.5,
            )

            conv_lo, conv_med, conv_hi, conv_n = bootstrap_probability_boundary(
                sub_rep, eta_values, "conventional_phase", "secure", target=0.5, seed=SEED + 200
            )
            uvif_lo, uvif_med, uvif_hi, uvif_n = bootstrap_probability_boundary(
                sub_rep, eta_values, "uvif_regime", "uvif_stable_secure", target=0.5, seed=SEED + 300
            )

            dp_conv = susceptibility(sub_sum["p_conventional_secure"].values, eta_values)
            dp_uvif = susceptibility(sub_sum["p_uvif_stable"].values, eta_values)
            dscore = susceptibility(sub_sum["uvif_score_mean"].values, eta_values)

            boundary_rows.append({
                "attack": float(attack),
                "drift": float(drift),
                "shots": int(shots),
                "eta_c_qber_threshold": eta_c_qber,
                "eta_c_finite_key_margin": eta_c_margin,
                "eta_c_conventional_psecure50": eta_c_conv_p50,
                "eta_c_conventional_boot_low": conv_lo,
                "eta_c_conventional_boot_med": conv_med,
                "eta_c_conventional_boot_high": conv_hi,
                "eta_c_conventional_boot_valid": conv_n,
                "eta_c_uvif_score50": eta_c_uvif_score,
                "eta_c_uvif_pstable50": eta_c_uvif_p50,
                "eta_c_uvif_boot_low": uvif_lo,
                "eta_c_uvif_boot_med": uvif_med,
                "eta_c_uvif_boot_high": uvif_hi,
                "eta_c_uvif_boot_valid": uvif_n,
                "uvif_minus_conventional_boundary": eta_c_uvif_p50 - eta_c_conv_p50 if not np.isnan(eta_c_uvif_p50) and not np.isnan(eta_c_conv_p50) else np.nan,
                "max_abs_dp_conventional_deta": float(np.nanmax(np.abs(dp_conv))),
                "max_abs_dp_uvif_deta": float(np.nanmax(np.abs(dp_uvif))),
                "max_abs_dscore_deta": float(np.nanmax(np.abs(dscore))),
                "eta_at_max_uvif_susceptibility": float(eta_values[int(np.nanargmax(np.abs(dp_uvif)))]),
            })
    return pd.DataFrame(boundary_rows).sort_values(["drift", "attack"]).reset_index(drop=True)

boundary_df = estimate_boundaries(endpoint_df, endpoint_summary_df, shots=DEFAULT_SHOTS)
boundary_df.to_csv(TAB_DIR / "table_uvif_boundary_estimates.csv", index=False)

log(f"[BOUNDARY] rows={len(boundary_df)}")
display(boundary_df.head(12))


In [ ]:
# ----------------------------
# Figure 1: conventional versus UVIF boundary estimates
# ----------------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 5.0))

for drift in DRIFT_GRID:
    sub = boundary_df[boundary_df["drift"] == float(drift)].sort_values("attack")

    axes[0].plot(
        sub["attack"],
        sub["eta_c_conventional_boot_med"],
        marker="o",
        lw=1.8,
        label=f"Finite-key conventional, drift={drift:.3f}",
    )
    axes[0].plot(
        sub["attack"],
        sub["eta_c_uvif_boot_med"],
        marker="s",
        lw=1.8,
        linestyle="--",
        label=f"UVIF-guided, drift={drift:.3f}",
    )

axes[0].set_xlabel("Adversarial pressure")
axes[0].set_ylabel(r"Boundary disturbance $\eta_c$")
axes[0].set_title("Secure-to-insecure boundary estimates")
axes[0].legend(fontsize=7)

for drift in DRIFT_GRID:
    sub = boundary_df[boundary_df["drift"] == float(drift)].sort_values("attack")
    axes[1].plot(
        sub["attack"],
        sub["uvif_minus_conventional_boundary"],
        marker="o",
        lw=1.8,
        label=f"drift={drift:.3f}",
    )

axes[1].axhline(0.0, linestyle="--", linewidth=1.2)
axes[1].set_xlabel("Adversarial pressure")
axes[1].set_ylabel(r"$\eta_{c,\mathrm{UVIF}}-\eta_{c,\mathrm{Conv.}}$")
axes[1].set_title("Resource-aware UVIF boundary shift")
axes[1].legend(fontsize=8)

plt.tight_layout()
fig_path = FIG_DIR / "fig_conventional_vs_uvif_boundary.png"
plt.savefig(fig_path, dpi=260, bbox_inches="tight")
plt.close()

log(f"[FIGURE] {fig_path}")


In [ ]:
# ----------------------------
# Figure 2: UVIF phase map and equilibrium-force decomposition
# ----------------------------
selected_drift = 0.010
selected_attack = 0.20

phase_map = endpoint_summary_df[
    (endpoint_summary_df["drift"] == selected_drift)
].pivot_table(index="attack", columns="eta", values="uvif_score_mean")

plt.figure(figsize=(9.5, 5.2))
plt.imshow(
    phase_map.values,
    aspect="auto",
    origin="lower",
    extent=[ETA_GRID.min(), ETA_GRID.max(), ATTACK_GRID.min(), ATTACK_GRID.max()],
)
plt.colorbar(label="Mean UVIF stability score")
plt.contour(
    ETA_GRID,
    ATTACK_GRID,
    phase_map.values,
    levels=[UVIF_BOUNDARY_LEVEL],
    linewidths=2.0,
)
plt.xlabel(r"Disturbance $\eta$")
plt.ylabel("Adversarial pressure")
plt.title(rf"UVIF stability phase map at drift={selected_drift:.3f}")
plt.tight_layout()
phase_map_path = FIG_DIR / "fig_uvif_stability_phase_map.png"
plt.savefig(phase_map_path, dpi=260, bbox_inches="tight")
plt.close()

force_cols = [
    "risk_force_mean",
    "uncertainty_force_mean",
    "complexity_force_mean",
    "resource_force_mean",
    "leakage_force_mean",
    "stability_reward_mean",
]
force_profile = endpoint_summary_df[
    (endpoint_summary_df["attack"] == selected_attack) &
    (endpoint_summary_df["drift"] == selected_drift)
].sort_values("eta")

plt.figure(figsize=(9.5, 5.2))
for col in force_cols:
    plt.plot(force_profile["eta"], force_profile[col], lw=2.0, label=col.replace("_mean", "").replace("_", " "))
plt.axvline(
    boundary_df[(boundary_df["attack"] == selected_attack) & (boundary_df["drift"] == selected_drift)]["eta_c_uvif_boot_med"].iloc[0],
    linestyle="--",
    linewidth=1.4,
)
plt.xlabel(r"Disturbance $\eta$")
plt.ylabel("Force magnitude / reward")
plt.title(rf"UVIF force decomposition at attack={selected_attack:.2f}, drift={selected_drift:.3f}")
plt.legend(fontsize=8)
plt.tight_layout()
force_path = FIG_DIR / "fig_uvif_force_decomposition.png"
plt.savefig(force_path, dpi=260, bbox_inches="tight")
plt.close()

log(f"[FIGURE] {phase_map_path}")
log(f"[FIGURE] {force_path}")


In [ ]:
# ----------------------------
# Disturbance-response profiles and susceptibility
# ----------------------------
profile_specs = [
    {"attack": 0.10, "drift": 0.005},
    {"attack": 0.20, "drift": 0.010},
    {"attack": 0.30, "drift": 0.020},
    {"attack": 0.35, "drift": 0.030},
]

profile_frames = []
for spec in profile_specs:
    sub = endpoint_summary_df[
        (endpoint_summary_df["attack"] == float(spec["attack"])) &
        (endpoint_summary_df["drift"] == float(spec["drift"]))
    ].sort_values("eta").copy()

    sub["dp_conventional_secure_deta"] = susceptibility(sub["p_conventional_secure"].values, sub["eta"].values)
    sub["dp_uvif_stable_deta"] = susceptibility(sub["p_uvif_stable"].values, sub["eta"].values)
    sub["d_uvif_score_deta"] = susceptibility(sub["uvif_score_mean"].values, sub["eta"].values)
    sub["profile"] = f"a={spec['attack']:.2f}, d={spec['drift']:.3f}"
    profile_frames.append(sub)

profiles_df = pd.concat(profile_frames, ignore_index=True)
profiles_df.to_csv(TAB_DIR / "table_disturbance_response_profiles.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.0))

for label, sub in profiles_df.groupby("profile"):
    sub = sub.sort_values("eta")
    axes[0].plot(sub["eta"], sub["p_conventional_secure"], lw=1.8, label=f"Conv. {label}")
    axes[0].plot(sub["eta"], sub["p_uvif_stable"], lw=1.8, linestyle="--", label=f"UVIF {label}")

axes[0].set_xlabel(r"Disturbance $\eta$")
axes[0].set_ylabel("Secure/stable probability")
axes[0].set_title("Disturbance-response profiles")
axes[0].legend(fontsize=7)

for label, sub in profiles_df.groupby("profile"):
    sub = sub.sort_values("eta")
    axes[1].plot(sub["eta"], np.abs(sub["dp_uvif_stable_deta"]), lw=2.0, label=label)

axes[1].set_xlabel(r"Disturbance $\eta$")
axes[1].set_ylabel(r"$|\partial p_{\mathrm{UVIF-stable}}/\partial \eta|$")
axes[1].set_title("UVIF transition susceptibility")
axes[1].legend(fontsize=8)

plt.tight_layout()
profile_path = FIG_DIR / "fig_disturbance_response_and_uvif_susceptibility.png"
plt.savefig(profile_path, dpi=260, bbox_inches="tight")
plt.close()

log(f"[PROFILE] rows={len(profiles_df)}")
log(f"[FIGURE] {profile_path}")


In [ ]:
# ----------------------------
# Shot-count and resource-robustness analysis
# ----------------------------
shot_summary_frames = []
shot_boundary_frames = []

for shots in SHOT_COUNTS:
    rng = np.random.default_rng(SEED + int(shots))
    rows = []

    for drift in DRIFT_GRID:
        for attack in ATTACK_GRID:
            for eta in ETA_GRID:
                df = simulate_endpoint_replicates(
                    eta=float(eta),
                    attack=float(attack),
                    drift=float(drift),
                    shots=int(shots),
                    n_repeats=max(80, N_REPEATS // 2),
                    rng=rng,
                )
                rows.append({
                    "eta": float(eta),
                    "attack": float(attack),
                    "drift": float(drift),
                    "shots": int(shots),
                    "p_conventional_secure": float((df["conventional_phase"] == "secure").mean()),
                    "p_uvif_stable": float((df["uvif_regime"] == "uvif_stable_secure").mean()),
                    "finite_key_margin_mean": float(df["finite_key_margin"].mean()),
                    "uvif_score_mean": float(df["uvif_score"].mean()),
                    "resource_force_mean": float(df["resource_force"].mean()),
                    "n_eff_mean": float(df["n_eff"].mean()),
                })

    shot_summary = pd.DataFrame(rows).sort_values(["shots", "drift", "attack", "eta"]).reset_index(drop=True)
    shot_summary_frames.append(shot_summary)

    for drift in DRIFT_GRID:
        for attack in ATTACK_GRID:
            sub = shot_summary[
                (shot_summary["drift"] == float(drift)) &
                (shot_summary["attack"] == float(attack))
            ].sort_values("eta")

            eta_values = sub["eta"].values
            shot_boundary_frames.append({
                "shots": int(shots),
                "attack": float(attack),
                "drift": float(drift),
                "eta_c_conventional_psecure50": interpolate_crossing(
                    eta_values, sub["p_conventional_secure"].values, target=0.5
                ),
                "eta_c_uvif_pstable50": interpolate_crossing(
                    eta_values, sub["p_uvif_stable"].values, target=0.5
                ),
                "eta_c_uvif_score50": interpolate_crossing(
                    eta_values, sub["uvif_score_mean"].values, target=UVIF_BOUNDARY_LEVEL
                ),
                "mean_resource_force": float(sub["resource_force_mean"].mean()),
                "mean_n_eff": float(sub["n_eff_mean"].mean()),
            })

shot_summary_df = pd.concat(shot_summary_frames, ignore_index=True)
shot_boundary_df = pd.DataFrame(shot_boundary_frames).sort_values(["shots", "drift", "attack"]).reset_index(drop=True)
shot_summary_df.to_csv(TAB_DIR / "table_shot_count_profiles.csv", index=False)
shot_boundary_df.to_csv(TAB_DIR / "table_shot_count_boundary_robustness.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.0))
selected_drift = 0.010

for shots in SHOT_COUNTS:
    sub = shot_boundary_df[
        (shot_boundary_df["shots"] == int(shots)) &
        (shot_boundary_df["drift"] == selected_drift)
    ].sort_values("attack")

    axes[0].plot(sub["attack"], sub["eta_c_conventional_psecure50"], marker="o", lw=1.8, label=f"Conv., {shots} shots")
    axes[0].plot(sub["attack"], sub["eta_c_uvif_pstable50"], marker="s", linestyle="--", lw=1.8, label=f"UVIF, {shots} shots")

axes[0].set_xlabel("Adversarial pressure")
axes[0].set_ylabel(r"Boundary disturbance $\eta_c$")
axes[0].set_title(rf"Shot-count boundary sensitivity at drift={selected_drift:.3f}")
axes[0].legend(fontsize=7)

shot_agg = shot_boundary_df.groupby("shots", as_index=False).agg(
    eta_c_conventional_mean=("eta_c_conventional_psecure50", "mean"),
    eta_c_conventional_sd=("eta_c_conventional_psecure50", "std"),
    eta_c_uvif_mean=("eta_c_uvif_pstable50", "mean"),
    eta_c_uvif_sd=("eta_c_uvif_pstable50", "std"),
    resource_force_mean=("mean_resource_force", "mean"),
)

axes[1].errorbar(
    shot_agg["shots"],
    shot_agg["eta_c_conventional_mean"],
    yerr=shot_agg["eta_c_conventional_sd"],
    marker="o",
    lw=1.8,
    capsize=4,
    label="Conventional finite-key",
)
axes[1].errorbar(
    shot_agg["shots"],
    shot_agg["eta_c_uvif_mean"],
    yerr=shot_agg["eta_c_uvif_sd"],
    marker="s",
    linestyle="--",
    lw=1.8,
    capsize=4,
    label="UVIF-guided",
)
axes[1].set_xlabel("Shot count")
axes[1].set_ylabel(r"Mean boundary disturbance $\eta_c$")
axes[1].set_title("Global resource robustness")
axes[1].legend(fontsize=8)

plt.tight_layout()
shot_path = FIG_DIR / "fig_shot_count_resource_robustness.png"
plt.savefig(shot_path, dpi=260, bbox_inches="tight")
plt.close()

shot_agg.to_csv(TAB_DIR / "table_shot_count_aggregate_robustness.csv", index=False)

log(f"[SHOT] profile rows={len(shot_summary_df)}, boundary rows={len(shot_boundary_df)}")
log(f"[FIGURE] {shot_path}")
display(shot_agg)


In [ ]:
# ----------------------------
# Phase-margin sensitivity analysis
# ----------------------------
PHASE_MARGIN_GRID = np.array([0.01, 0.02, 0.03, 0.05], dtype=float)
phase_rows = []

for phase_margin in PHASE_MARGIN_GRID:
    temp = endpoint_df.copy()
    temp["phase_margin"] = float(phase_margin)
    temp["conv_secure_pm"] = (temp["psi"] > float(phase_margin)).astype(float)

    grouped = temp.groupby(["phase_margin", "drift", "attack", "eta"], as_index=False).agg(
        p_conventional_secure=("conv_secure_pm", "mean"),
        p_uvif_stable=("uvif_regime", lambda s: float(np.mean(np.asarray(s) == "uvif_stable_secure"))),
        uvif_score_mean=("uvif_score", "mean"),
    )

    for drift in DRIFT_GRID:
        for attack in ATTACK_GRID:
            sub = grouped[
                (grouped["drift"] == float(drift)) &
                (grouped["attack"] == float(attack))
            ].sort_values("eta")
            eta_values = sub["eta"].values
            phase_rows.append({
                "phase_margin": float(phase_margin),
                "attack": float(attack),
                "drift": float(drift),
                "eta_c_conventional_psecure50": interpolate_crossing(eta_values, sub["p_conventional_secure"].values, 0.5),
                "eta_c_uvif_pstable50": interpolate_crossing(eta_values, sub["p_uvif_stable"].values, 0.5),
                "eta_c_uvif_score50": interpolate_crossing(eta_values, sub["uvif_score_mean"].values, UVIF_BOUNDARY_LEVEL),
            })

phase_boundary_df = pd.DataFrame(phase_rows).sort_values(["phase_margin", "drift", "attack"]).reset_index(drop=True)

baseline_pm = PHASE_MARGIN
baseline = (
    phase_boundary_df[phase_boundary_df["phase_margin"] == baseline_pm]
    [["attack", "drift", "eta_c_conventional_psecure50", "eta_c_uvif_pstable50"]]
    .rename(columns={
        "eta_c_conventional_psecure50": "eta_c_conventional_baseline",
        "eta_c_uvif_pstable50": "eta_c_uvif_baseline",
    })
)

phase_delta_df = phase_boundary_df.merge(baseline, on=["attack", "drift"], how="left")
phase_delta_df["delta_conventional"] = phase_delta_df["eta_c_conventional_psecure50"] - phase_delta_df["eta_c_conventional_baseline"]
phase_delta_df["delta_uvif"] = phase_delta_df["eta_c_uvif_pstable50"] - phase_delta_df["eta_c_uvif_baseline"]

phase_summary_df = phase_delta_df.groupby("phase_margin", as_index=False).agg(
    conventional_delta_mean=("delta_conventional", "mean"),
    conventional_delta_sd=("delta_conventional", "std"),
    uvif_delta_mean=("delta_uvif", "mean"),
    uvif_delta_sd=("delta_uvif", "std"),
)

phase_boundary_df.to_csv(TAB_DIR / "table_phase_margin_boundary_sensitivity.csv", index=False)
phase_summary_df.to_csv(TAB_DIR / "table_phase_margin_sensitivity_summary.csv", index=False)

plt.figure(figsize=(8.5, 5.0))
plt.errorbar(
    phase_summary_df["phase_margin"],
    phase_summary_df["conventional_delta_mean"],
    yerr=phase_summary_df["conventional_delta_sd"],
    marker="o",
    lw=1.8,
    capsize=4,
    label="Conventional finite-key boundary",
)
plt.errorbar(
    phase_summary_df["phase_margin"],
    phase_summary_df["uvif_delta_mean"],
    yerr=phase_summary_df["uvif_delta_sd"],
    marker="s",
    linestyle="--",
    lw=1.8,
    capsize=4,
    label="UVIF-guided boundary",
)
plt.axhline(0.0, linestyle="--", linewidth=1.2)
plt.xlabel("Descriptive phase margin")
plt.ylabel(r"Boundary shift relative to baseline $\Delta\eta_c$")
plt.title("Sensitivity to phase-margin convention")
plt.legend()
plt.tight_layout()
phase_path = FIG_DIR / "fig_phase_margin_sensitivity.png"
plt.savefig(phase_path, dpi=260, bbox_inches="tight")
plt.close()

log(f"[PHASE] rows={len(phase_boundary_df)}")
log(f"[FIGURE] {phase_path}")
display(phase_summary_df)


In [ ]:
# ----------------------------
# Time-resolved scenario analysis and early-warning indicators
# ----------------------------
SCENARIO_REPEATS = 90
scenario_rng = np.random.default_rng(SEED + 7000)

trajectory_frames = []
trial_rows = []

for sc in SCENARIOS:
    seeds = scenario_rng.integers(0, 2**32 - 1, size=SCENARIO_REPEATS)
    for repeat, seed in enumerate(seeds):
        traj = simulate_trajectory(
            eta=float(sc["eta"]),
            attack=float(sc["attack"]),
            drift=float(sc["drift"]),
            shots=int(sc["shots"]),
            time_horizon=TIME_HORIZON,
            seed=int(seed),
        )
        traj["scenario"] = sc["name"]
        traj["repeat"] = int(repeat)
        trajectory_frames.append(traj)

        trial_rows.append({
            "scenario": sc["name"],
            "repeat": int(repeat),
            "eta": float(sc["eta"]),
            "attack": float(sc["attack"]),
            "drift": float(sc["drift"]),
            "shots": int(sc["shots"]),
            "final_qber_obs": float(traj["qber_obs"].iloc[-1]),
            "final_finite_key_margin": float(traj["finite_key_margin"].iloc[-1]),
            "final_psi": float(traj["psi"].iloc[-1]),
            "final_uvif_energy": float(traj["uvif_energy"].iloc[-1]),
            "final_uvif_score": float(traj["uvif_score"].iloc[-1]),
            "final_conventional_phase": str(traj["conventional_phase"].iloc[-1]),
            "final_uvif_regime": str(traj["uvif_regime"].iloc[-1]),
            "first_conventional_transition_time": first_transition_time(traj["psi"].values, 0.0, "below"),
            "first_uvif_transition_time": first_transition_time(traj["uvif_score"].values, UVIF_BOUNDARY_LEVEL, "below"),
            "peak_early_warning_index": float(traj["early_warning_index"].max()),
            "mean_early_warning_index": float(traj["early_warning_index"].mean()),
        })

scenario_traj_df = pd.concat(trajectory_frames, ignore_index=True)
scenario_trial_df = pd.DataFrame(trial_rows)

scenario_time_summary_df = scenario_traj_df.groupby(["scenario", "t"], as_index=False).agg(
    qber_mean=("qber_obs", "mean"),
    qber_q05=("qber_obs", lambda x: float(np.quantile(x, 0.05))),
    qber_q50=("qber_obs", lambda x: float(np.quantile(x, 0.50))),
    qber_q95=("qber_obs", lambda x: float(np.quantile(x, 0.95))),
    psi_mean=("psi", "mean"),
    psi_q05=("psi", lambda x: float(np.quantile(x, 0.05))),
    psi_q50=("psi", lambda x: float(np.quantile(x, 0.50))),
    psi_q95=("psi", lambda x: float(np.quantile(x, 0.95))),
    uvif_score_mean=("uvif_score", "mean"),
    uvif_score_q05=("uvif_score", lambda x: float(np.quantile(x, 0.05))),
    uvif_score_q50=("uvif_score", lambda x: float(np.quantile(x, 0.50))),
    uvif_score_q95=("uvif_score", lambda x: float(np.quantile(x, 0.95))),
    early_warning_mean=("early_warning_index", "mean"),
    early_warning_q95=("early_warning_index", lambda x: float(np.quantile(x, 0.95))),
    p_conventional_secure=("conventional_phase", lambda s: float(np.mean(np.asarray(s) == "secure"))),
    p_uvif_stable=("uvif_regime", lambda s: float(np.mean(np.asarray(s) == "uvif_stable_secure"))),
)

def median_valid_time(values):
    arr = np.asarray(values, dtype=float)
    valid = arr[arr >= 0]
    if len(valid) == 0:
        return -1.0
    return float(np.median(valid))

scenario_summary_df = scenario_trial_df.groupby("scenario", as_index=False).agg(
    eta=("eta", "first"),
    attack=("attack", "first"),
    drift=("drift", "first"),
    shots=("shots", "first"),
    final_qber_mean=("final_qber_obs", "mean"),
    final_margin_mean=("final_finite_key_margin", "mean"),
    final_psi_mean=("final_psi", "mean"),
    final_uvif_score_mean=("final_uvif_score", "mean"),
    p_final_conventional_secure=("final_conventional_phase", lambda s: float(np.mean(np.asarray(s) == "secure"))),
    p_final_conventional_insecure=("final_conventional_phase", lambda s: float(np.mean(np.asarray(s) == "insecure"))),
    p_final_uvif_stable=("final_uvif_regime", lambda s: float(np.mean(np.asarray(s) == "uvif_stable_secure"))),
    p_final_uvif_unstable=("final_uvif_regime", lambda s: float(np.mean(np.asarray(s) == "uvif_unstable_insecure"))),
    p_ever_conventional_transition=("first_conventional_transition_time", lambda x: float(np.mean(np.asarray(x) >= 0))),
    p_ever_uvif_transition=("first_uvif_transition_time", lambda x: float(np.mean(np.asarray(x) >= 0))),
    median_first_conventional_transition_time=("first_conventional_transition_time", median_valid_time),
    median_first_uvif_transition_time=("first_uvif_transition_time", median_valid_time),
    peak_early_warning_mean=("peak_early_warning_index", "mean"),
    mean_early_warning_mean=("mean_early_warning_index", "mean"),
)

scenario_order = [sc["name"] for sc in SCENARIOS]
for df in [scenario_time_summary_df, scenario_summary_df]:
    df["scenario"] = pd.Categorical(df["scenario"], categories=scenario_order, ordered=True)
scenario_time_summary_df = scenario_time_summary_df.sort_values(["scenario", "t"]).reset_index(drop=True)
scenario_summary_df = scenario_summary_df.sort_values("scenario").reset_index(drop=True)

scenario_traj_df.to_csv(TAB_DIR / "table_scenario_trajectories_full.csv", index=False)
scenario_time_summary_df.to_csv(TAB_DIR / "table_scenario_time_summary.csv", index=False)
scenario_summary_df.to_csv(TAB_DIR / "table_scenario_summary.csv", index=False)

log(f"[SCENARIO] trajectory rows={len(scenario_traj_df):,}")
display(scenario_summary_df)


In [ ]:
# ----------------------------
# Figure 6: temporal trajectories and early-warning behavior
# ----------------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 5.0))

for scenario in scenario_order:
    sub = scenario_time_summary_df[scenario_time_summary_df["scenario"] == scenario].sort_values("t")
    t = sub["t"].to_numpy(dtype=float)

    axes[0].plot(t, sub["psi_mean"], lw=2.0, label=scenario)
    axes[0].fill_between(t, sub["psi_q05"].values, sub["psi_q95"].values, alpha=0.12)

axes[0].axhline(0.0, linestyle="--", linewidth=1.2)
axes[0].set_xlabel("Time index")
axes[0].set_ylabel(r"Finite-key order parameter $\psi$")
axes[0].set_title("Conventional finite-key security trajectories")
axes[0].legend(fontsize=7)

for scenario in scenario_order:
    sub = scenario_time_summary_df[scenario_time_summary_df["scenario"] == scenario].sort_values("t")
    t = sub["t"].to_numpy(dtype=float)

    axes[1].plot(t, sub["uvif_score_mean"], lw=2.0, label=scenario)
    axes[1].fill_between(t, sub["uvif_score_q05"].values, sub["uvif_score_q95"].values, alpha=0.12)

axes[1].axhline(UVIF_BOUNDARY_LEVEL, linestyle="--", linewidth=1.2)
axes[1].set_xlabel("Time index")
axes[1].set_ylabel("UVIF stability score")
axes[1].set_title("UVIF-guided stability trajectories")
axes[1].legend(fontsize=7)

plt.tight_layout()
traj_path = FIG_DIR / "fig_temporal_security_trajectories.png"
plt.savefig(traj_path, dpi=260, bbox_inches="tight")
plt.close()

plt.figure(figsize=(9.5, 5.0))
for scenario in scenario_order:
    sub = scenario_time_summary_df[scenario_time_summary_df["scenario"] == scenario].sort_values("t")
    plt.plot(sub["t"], sub["early_warning_mean"], lw=2.0, label=scenario)
plt.xlabel("Time index")
plt.ylabel("Early-warning index")
plt.title("Trajectory-based early-warning indicator")
plt.legend(fontsize=8)
plt.tight_layout()
warning_path = FIG_DIR / "fig_early_warning_indicator.png"
plt.savefig(warning_path, dpi=260, bbox_inches="tight")
plt.close()

log(f"[FIGURE] {traj_path}")
log(f"[FIGURE] {warning_path}")


In [ ]:
# ----------------------------
# Final compact tables and manuscript-ready output summary
# ----------------------------
boundary_compact = boundary_df.loc[:, [
    "attack",
    "drift",
    "eta_c_qber_threshold",
    "eta_c_finite_key_margin",
    "eta_c_conventional_boot_low",
    "eta_c_conventional_boot_med",
    "eta_c_conventional_boot_high",
    "eta_c_uvif_boot_low",
    "eta_c_uvif_boot_med",
    "eta_c_uvif_boot_high",
    "uvif_minus_conventional_boundary",
    "max_abs_dp_uvif_deta",
]].copy()

force_summary = endpoint_summary_df.groupby(["attack", "drift"], as_index=False).agg(
    risk_force_mean=("risk_force_mean", "mean"),
    uncertainty_force_mean=("uncertainty_force_mean", "mean"),
    complexity_force_mean=("complexity_force_mean", "mean"),
    resource_force_mean=("resource_force_mean", "mean"),
    leakage_force_mean=("leakage_force_mean", "mean"),
    stability_reward_mean=("stability_reward_mean", "mean"),
    uvif_score_mean=("uvif_score_mean", "mean"),
)

resource_robustness = shot_agg.copy()
phase_margin_summary = phase_summary_df.copy()
scenario_compact = scenario_summary_df.copy()

boundary_compact.to_csv(TAB_DIR / "manuscript_table_boundary_comparison.csv", index=False)
force_summary.to_csv(TAB_DIR / "manuscript_table_uvif_force_summary.csv", index=False)
resource_robustness.to_csv(TAB_DIR / "manuscript_table_resource_robustness.csv", index=False)
phase_margin_summary.to_csv(TAB_DIR / "manuscript_table_phase_margin_summary.csv", index=False)
scenario_compact.to_csv(TAB_DIR / "manuscript_table_temporal_scenarios.csv", index=False)

generated_files = sorted([str(p) for p in PROJECT_ROOT.rglob("*") if p.is_file()])

summary_lines = [
    "UVIF-guided finite-key BB84 boundary dynamics: outputs summary",
    "=" * 78,
    f"Project: {PROJECT_NAME}",
    f"Run ID: {RUN_ID}",
    f"Project root: {PROJECT_ROOT}",
    "",
    "Notebook framing:",
    "- Conventional finite-key BB84-inspired security is represented through QBER, finite-key secrecy margin, and a security order parameter.",
    "- UVIF extends the baseline by combining risk, uncertainty, implementation complexity, resource pressure, leakage burden, and stability reward into an operational free-energy functional.",
    "- Secure-to-insecure transition is estimated using both conventional finite-key probability and UVIF-guided stability probability.",
    "",
    "Main tables:",
    f"- {TAB_DIR / 'manuscript_table_boundary_comparison.csv'}",
    f"- {TAB_DIR / 'manuscript_table_uvif_force_summary.csv'}",
    f"- {TAB_DIR / 'manuscript_table_resource_robustness.csv'}",
    f"- {TAB_DIR / 'manuscript_table_phase_margin_summary.csv'}",
    f"- {TAB_DIR / 'manuscript_table_temporal_scenarios.csv'}",
    "",
    "Main figures:",
    f"- {FIG_DIR / 'fig_conventional_vs_uvif_boundary.png'}",
    f"- {FIG_DIR / 'fig_uvif_stability_phase_map.png'}",
    f"- {FIG_DIR / 'fig_uvif_force_decomposition.png'}",
    f"- {FIG_DIR / 'fig_disturbance_response_and_uvif_susceptibility.png'}",
    f"- {FIG_DIR / 'fig_shot_count_resource_robustness.png'}",
    f"- {FIG_DIR / 'fig_phase_margin_sensitivity.png'}",
    f"- {FIG_DIR / 'fig_temporal_security_trajectories.png'}",
    f"- {FIG_DIR / 'fig_early_warning_indicator.png'}",
    "",
    "Suggested manuscript mapping:",
    "- Methodology: use endpoint_summary_uvif.csv, table_uvif_boundary_estimates.csv, and table_shot_count_boundary_robustness.csv.",
    "- Results: use the conventional-vs-UVIF boundary figure, UVIF phase map, force decomposition, resource robustness, and temporal trajectory figures.",
    "- Limitations: emphasize that the model is BB84-inspired and operational/simulation-based, not a replacement for composable security proofs.",
    "",
    "Generated file inventory:",
]
summary_lines.extend(f"- {path}" for path in generated_files)

summary_path = OUT_DIR / "outputs_summary.txt"
with open(summary_path, "w", encoding="utf-8") as f:
    f.write("\n".join(summary_lines))

log(f"[SUMMARY] {summary_path}")
log("[DONE] Notebook run complete.")

print("\n[BOUNDARY COMPACT]")
display(boundary_compact.head(12))

print("\n[RESOURCE ROBUSTNESS]")
display(resource_robustness)

print("\n[SCENARIO SUMMARY]")
display(scenario_compact)


## Array-targeted reviewer-proofing upgrade

This addendum strengthens the notebook for an interdisciplinary intelligent-systems journal such as **Array**.  The additional analyses are designed to make the computational contribution clearer: UVIF is treated as an operational intelligence and decision-support layer placed above a BB84-inspired finite-key simulation, rather than as a replacement for composable quantum-security proofs.

The upgrade adds four reviewer-facing elements:

1. an ablation comparison of QBER-only, finite-key margin, UVIF without resource pressure, and full UVIF;
2. bootstrap confidence summaries for boundary estimates;
3. an actionable decision-support regime table; and
4. a compact manuscript-ready table of main findings.


In [ ]:
# ============================================================
# Array-targeted upgrade 1: ablation study for operational intelligence claims
# ============================================================
# Purpose:
#   Compare the proposed full UVIF layer against progressively richer baselines:
#   (A1) conventional QBER thresholding only,
#   (A2) finite-key secrecy-margin assessment,
#   (A3) UVIF without resource pressure,
#   (A4) full resource-aware UVIF.
# This makes the added value of UVIF explicit for reviewers.

ABLATED_WEIGHTS = dict(UVIF_WEIGHTS)
ABLATED_WEIGHTS["resource"] = 0.0

endpoint_ablation_df = endpoint_df.copy()
endpoint_ablation_df["secure_qber_only"] = (endpoint_ablation_df["qber_obs"] <= QBER_SECURITY_THRESHOLD).astype(float)
endpoint_ablation_df["secure_finite_key_margin"] = (endpoint_ablation_df["finite_key_margin"] > 0.0).astype(float)

# Recompute a no-resource UVIF score using already generated force terms.
def uvif_energy_no_resource_from_row(row):
    return float(
        ABLATED_WEIGHTS["risk"] * row["risk_force"]
        + ABLATED_WEIGHTS["uncertainty"] * row["uncertainty_force"]
        + ABLATED_WEIGHTS["complexity"] * row["complexity_force"]
        + ABLATED_WEIGHTS["resource"] * row["resource_force"]
        + ABLATED_WEIGHTS["leakage"] * row["leakage_force"]
        - ABLATED_WEIGHTS["stability_reward"] * row["stability_reward"]
    )

endpoint_ablation_df["uvif_energy_no_resource"] = endpoint_ablation_df.apply(uvif_energy_no_resource_from_row, axis=1)
endpoint_ablation_df["uvif_score_no_resource"] = 1.0 / (1.0 + np.exp(endpoint_ablation_df["uvif_energy_no_resource"]))
endpoint_ablation_df["secure_uvif_no_resource"] = (endpoint_ablation_df["uvif_score_no_resource"] >= UVIF_BOUNDARY_LEVEL).astype(float)
endpoint_ablation_df["secure_full_uvif"] = (endpoint_ablation_df["uvif_score"] >= UVIF_BOUNDARY_LEVEL).astype(float)

ablation_probability_df = endpoint_ablation_df.groupby(["drift", "attack", "eta"], as_index=False).agg(
    p_qber_only=("secure_qber_only", "mean"),
    p_finite_key_margin=("secure_finite_key_margin", "mean"),
    p_uvif_no_resource=("secure_uvif_no_resource", "mean"),
    p_full_uvif=("secure_full_uvif", "mean"),
    qber_obs_mean=("qber_obs", "mean"),
    finite_key_margin_mean=("finite_key_margin", "mean"),
    uvif_score_no_resource_mean=("uvif_score_no_resource", "mean"),
    uvif_score_full_mean=("uvif_score", "mean"),
    resource_force_mean=("resource_force", "mean"),
)

ablation_probability_df.to_csv(TAB_DIR / "table_array_ablation_probability_profiles.csv", index=False)

ABLATION_MODELS = {
    "QBER only": "secure_qber_only",
    "Finite-key margin": "secure_finite_key_margin",
    "UVIF without resource": "secure_uvif_no_resource",
    "Full UVIF": "secure_full_uvif",
}


def bootstrap_binary_boundary(df, eta_values, binary_col, target=0.5, n_boot=BOOTSTRAPS, seed=SEED):
    rng = np.random.default_rng(seed)
    grouped = {float(eta): group[binary_col].to_numpy(dtype=float) for eta, group in df.groupby("eta")}
    eta_values = np.asarray(eta_values, dtype=float)
    samples = []

    for _ in range(n_boot):
        probs = []
        for eta in eta_values:
            values = grouped.get(float(eta))
            if values is None or len(values) == 0:
                probs.append(np.nan)
            else:
                boot_values = rng.choice(values, size=len(values), replace=True)
                probs.append(float(np.nanmean(boot_values)))
        boundary = interpolate_crossing(eta_values, np.asarray(probs), target=target)
        if not np.isnan(boundary):
            samples.append(boundary)

    if len(samples) == 0:
        return np.nan, np.nan, np.nan, 0
    low, med, high = np.quantile(samples, [0.025, 0.50, 0.975])
    return float(low), float(med), float(high), int(len(samples))

ablation_rows = []
for drift in DRIFT_GRID:
    for attack in ATTACK_GRID:
        sub = endpoint_ablation_df[
            (endpoint_ablation_df["drift"] == float(drift)) &
            (endpoint_ablation_df["attack"] == float(attack)) &
            (endpoint_ablation_df["shots"] == int(DEFAULT_SHOTS))
        ].copy()
        eta_values = np.sort(sub["eta"].unique())
        for model_name, col in ABLATION_MODELS.items():
            grouped = sub.groupby("eta", as_index=False)[col].mean().sort_values("eta")
            eta_c = interpolate_crossing(grouped["eta"].values, grouped[col].values, target=0.5)
            low, med, high, n_valid = bootstrap_binary_boundary(
                sub, eta_values, col, target=0.5, n_boot=BOOTSTRAPS, seed=SEED + abs(hash(model_name)) % 10000
            )
            ablation_rows.append({
                "model": model_name,
                "attack": float(attack),
                "drift": float(drift),
                "eta_c_probability50": eta_c,
                "eta_c_boot_low": low,
                "eta_c_boot_median": med,
                "eta_c_boot_high": high,
                "eta_c_ci_width": high - low if not np.isnan(high) and not np.isnan(low) else np.nan,
                "n_valid_bootstrap_boundaries": int(n_valid),
            })

ablation_boundary_df = pd.DataFrame(ablation_rows).sort_values(["drift", "attack", "model"]).reset_index(drop=True)
ablation_boundary_df.to_csv(TAB_DIR / "manuscript_table_array_ablation_boundary_comparison.csv", index=False)

# Manuscript-ready ablation summary by model.
ablation_model_summary = ablation_boundary_df.groupby("model", as_index=False).agg(
    mean_eta_c=("eta_c_boot_median", "mean"),
    sd_eta_c=("eta_c_boot_median", "std"),
    mean_ci_width=("eta_c_ci_width", "mean"),
    valid_boundary_count=("n_valid_bootstrap_boundaries", lambda x: int(np.sum(np.asarray(x) > 0))),
)
ablation_model_summary.to_csv(TAB_DIR / "manuscript_table_array_ablation_model_summary.csv", index=False)

log("[ARRAY-UPGRADE] Ablation profiles and boundary summaries saved.")
print("[ABLATION MODEL SUMMARY]")
display(ablation_model_summary)


In [ ]:
# ============================================================
# Array-targeted upgrade 2: ablation figure for reviewer-facing comparison
# ============================================================
selected_drift_for_ablation = 0.010 if 0.010 in set(np.round(DRIFT_GRID, 3)) else float(DRIFT_GRID[0])

plt.figure(figsize=(9.8, 5.4))
for model_name in ["QBER only", "Finite-key margin", "UVIF without resource", "Full UVIF"]:
    sub = ablation_boundary_df[
        (ablation_boundary_df["drift"] == selected_drift_for_ablation) &
        (ablation_boundary_df["model"] == model_name)
    ].sort_values("attack")
    plt.plot(sub["attack"], sub["eta_c_boot_median"], marker="o", linewidth=2.0, label=model_name)

plt.xlabel("Adversarial pressure")
plt.ylabel(r"Boundary disturbance $\eta_c$")
plt.title(f"Ablation of operational boundary estimation at drift={selected_drift_for_ablation:.3f}")
plt.legend(fontsize=8)
plt.tight_layout()
ablation_fig_path = FIG_DIR / "fig_array_ablation_boundary_comparison.png"
plt.savefig(ablation_fig_path, dpi=260, bbox_inches="tight")
plt.close()

log(f"[ARRAY-UPGRADE] Ablation boundary figure saved: {ablation_fig_path}")


In [ ]:
# ============================================================
# Array-targeted upgrade 3: decision-support regimes and action table
# ============================================================
# This section translates UVIF and finite-key outputs into actionable operational states.
# It is intended for Array-style intelligent decision-system framing.

def assign_decision_regime(row):
    score = float(row["uvif_score_mean"])
    margin = float(row["finite_key_margin_mean"])
    resource = float(row["resource_force_mean"])

    if score >= 0.62 and margin > 0.0 and resource < 0.20:
        return "secure_operate"
    if score >= 0.50 and margin > -0.05:
        return "watch_adapt"
    if score >= 0.38:
        return "warning_reduce_risk"
    return "insecure_suspend_or_rekey"

regime_action_map = pd.DataFrame([
    {
        "decision_regime": "secure_operate",
        "interpretation": "Stable operational security margin with acceptable UVIF free-energy balance.",
        "recommended_action": "Continue operation; monitor QBER, finite-key margin, and UVIF score.",
        "manuscript_use": "Represents the nominal secure state under the proposed operational-intelligence layer.",
    },
    {
        "decision_regime": "watch_adapt",
        "interpretation": "Security remains acceptable but the boundary is approaching under disturbance, drift, or finite resources.",
        "recommended_action": "Increase monitoring frequency, refresh calibration, or adjust resource allocation.",
        "manuscript_use": "Shows how UVIF creates an intermediate adaptive decision state absent from hard QBER thresholding.",
    },
    {
        "decision_regime": "warning_reduce_risk",
        "interpretation": "Transition-like region with elevated free energy, weakened margin, or resource pressure.",
        "recommended_action": "Reduce key-generation exposure, increase shot count if possible, recalibrate, or trigger rekeying policy.",
        "manuscript_use": "Provides early-warning and risk-mitigation functionality.",
    },
    {
        "decision_regime": "insecure_suspend_or_rekey",
        "interpretation": "Operational security state is unstable or insecure under the UVIF-guided assessment.",
        "recommended_action": "Suspend key use, re-estimate channel conditions, rekey, or reject the block.",
        "manuscript_use": "Represents a conservative operational response to high disturbance or adversarial pressure.",
    },
])
regime_action_map.to_csv(TAB_DIR / "manuscript_table_array_decision_support_regime_actions.csv", index=False)

endpoint_decision_df = endpoint_summary_df.copy()
endpoint_decision_df["decision_regime"] = endpoint_decision_df.apply(assign_decision_regime, axis=1)
endpoint_decision_df.to_csv(TAB_DIR / "table_array_decision_regime_profiles.csv", index=False)

regime_distribution = endpoint_decision_df.groupby(["attack", "drift", "decision_regime"], as_index=False).agg(
    n_eta_points=("eta", "count"),
    eta_min=("eta", "min"),
    eta_max=("eta", "max"),
    uvif_score_mean=("uvif_score_mean", "mean"),
    finite_key_margin_mean=("finite_key_margin_mean", "mean"),
    resource_force_mean=("resource_force_mean", "mean"),
)
regime_distribution.to_csv(TAB_DIR / "manuscript_table_array_decision_regime_distribution.csv", index=False)

log("[ARRAY-UPGRADE] Decision-support regime tables saved.")
print("[DECISION-SUPPORT ACTION TABLE]")
display(regime_action_map)


In [ ]:
# ============================================================
# Array-targeted upgrade 4: bootstrap confidence and main-finding summaries
# ============================================================
boundary_confidence_summary = boundary_df.copy()
boundary_confidence_summary["conventional_ci_width"] = (
    boundary_confidence_summary["eta_c_conventional_boot_high"] - boundary_confidence_summary["eta_c_conventional_boot_low"]
)
boundary_confidence_summary["uvif_ci_width"] = (
    boundary_confidence_summary["eta_c_uvif_boot_high"] - boundary_confidence_summary["eta_c_uvif_boot_low"]
)
boundary_confidence_summary["uvif_ci_narrower_than_conventional"] = (
    boundary_confidence_summary["uvif_ci_width"] < boundary_confidence_summary["conventional_ci_width"]
)

boundary_confidence_summary.to_csv(TAB_DIR / "manuscript_table_array_bootstrap_boundary_confidence.csv", index=False)

mean_boundary_shift = float(np.nanmean(boundary_confidence_summary["uvif_minus_conventional_boundary"]))
mean_uvif_ci_width = float(np.nanmean(boundary_confidence_summary["uvif_ci_width"]))
mean_conv_ci_width = float(np.nanmean(boundary_confidence_summary["conventional_ci_width"]))
share_uvif_narrower = float(np.nanmean(boundary_confidence_summary["uvif_ci_narrower_than_conventional"].astype(float)))

# Resource sensitivity from the shot-count robustness table.
resource_sensitivity = shot_boundary_df.groupby("shots", as_index=False).agg(
    mean_eta_c_conventional=("eta_c_conventional_psecure50", "mean"),
    mean_eta_c_uvif=("eta_c_uvif_pstable50", "mean"),
    mean_eta_c_uvif_score=("eta_c_uvif_score50", "mean"),
)
resource_sensitivity.to_csv(TAB_DIR / "manuscript_table_array_resource_sensitivity_summary.csv", index=False)

try:
    low_shots = resource_sensitivity.loc[resource_sensitivity["shots"] == int(SHOT_COUNTS.min()), "mean_eta_c_uvif"].iloc[0]
    high_shots = resource_sensitivity.loc[resource_sensitivity["shots"] == int(SHOT_COUNTS.max()), "mean_eta_c_uvif"].iloc[0]
    uvif_resource_gain = float(high_shots - low_shots)
except Exception:
    uvif_resource_gain = np.nan

# Early-warning summary based on scenario trajectories.
earliest_warning_by_scenario = scenario_summary_df[[
    "scenario", "eta", "attack", "drift", "shots",
    "mean_time_to_conventional_insecure", "mean_time_to_uvif_unstable", "mean_time_to_warning"
]].copy()
earliest_warning_by_scenario["warning_lead_vs_conventional"] = (
    earliest_warning_by_scenario["mean_time_to_conventional_insecure"] - earliest_warning_by_scenario["mean_time_to_warning"]
)
earliest_warning_by_scenario.to_csv(TAB_DIR / "manuscript_table_array_early_warning_summary.csv", index=False)

main_findings = pd.DataFrame([
    {
        "finding_block": "Boundary estimation",
        "main_result": "Full UVIF provides a distinct operational boundary relative to conventional finite-key/QBER assessment.",
        "quantitative_indicator": f"Mean UVIF-minus-conventional boundary shift = {mean_boundary_shift:.4f}",
        "recommended_manuscript_location": "Results: Operational Security Boundary Across Disturbance Conditions",
    },
    {
        "finding_block": "Uncertainty quantification",
        "main_result": "Bootstrap intervals are available for the conventional and UVIF boundaries.",
        "quantitative_indicator": f"Mean CI width: conventional={mean_conv_ci_width:.4f}, UVIF={mean_uvif_ci_width:.4f}; UVIF narrower share={share_uvif_narrower:.2f}",
        "recommended_manuscript_location": "Methodology: Bootstrap Uncertainty Quantification; Results: Boundary Robustness",
    },
    {
        "finding_block": "Ablation evidence",
        "main_result": "The added resource-aware UVIF terms can be isolated against QBER-only, finite-key, and no-resource UVIF variants.",
        "quantitative_indicator": "See manuscript_table_array_ablation_boundary_comparison.csv",
        "recommended_manuscript_location": "Results: Comparative Interpretation Against Conventional QBER Thresholding",
    },
    {
        "finding_block": "Resource robustness",
        "main_result": "Shot-count sensitivity explicitly links finite resources to the estimated operational security boundary.",
        "quantitative_indicator": f"Mean UVIF boundary gain from lowest to highest shot count = {uvif_resource_gain:.4f}",
        "recommended_manuscript_location": "Results: Sensitivity to Resource Availability",
    },
    {
        "finding_block": "Decision support",
        "main_result": "UVIF produces intermediate watch and warning regimes rather than only binary secure/insecure labels.",
        "quantitative_indicator": "See manuscript_table_array_decision_support_regime_actions.csv",
        "recommended_manuscript_location": "Results: Predictive Stability and Early-Warning Indicators",
    },
])
main_findings.to_csv(TAB_DIR / "manuscript_table_array_main_findings.csv", index=False)

log("[ARRAY-UPGRADE] Bootstrap confidence, early-warning, resource-sensitivity, and main-finding tables saved.")
print("[MAIN FINDINGS FOR MANUSCRIPT]")
display(main_findings)


In [ ]:
# ============================================================
# Array-targeted upgrade 5: additional output summary for submission planning
# ============================================================
array_upgrade_files = [
    TAB_DIR / "table_array_ablation_probability_profiles.csv",
    TAB_DIR / "manuscript_table_array_ablation_boundary_comparison.csv",
    TAB_DIR / "manuscript_table_array_ablation_model_summary.csv",
    FIG_DIR / "fig_array_ablation_boundary_comparison.png",
    TAB_DIR / "manuscript_table_array_decision_support_regime_actions.csv",
    TAB_DIR / "table_array_decision_regime_profiles.csv",
    TAB_DIR / "manuscript_table_array_decision_regime_distribution.csv",
    TAB_DIR / "manuscript_table_array_bootstrap_boundary_confidence.csv",
    TAB_DIR / "manuscript_table_array_resource_sensitivity_summary.csv",
    TAB_DIR / "manuscript_table_array_early_warning_summary.csv",
    TAB_DIR / "manuscript_table_array_main_findings.csv",
]

array_summary_lines = [
    "Array-targeted UVIF-BB84 notebook upgrade summary",
    "=" * 72,
    f"Project: {PROJECT_NAME}",
    f"Run ID: {RUN_ID}",
    f"Project root: {PROJECT_ROOT}",
    "",
    "Purpose of this upgrade:",
    "- Reframe UVIF as an operational intelligence and decision-support layer for finite-key BB84-inspired communications.",
    "- Add reviewer-facing ablation evidence against QBER-only, finite-key margin, and no-resource UVIF variants.",
    "- Add bootstrap confidence summaries and manuscript-ready main findings.",
    "- Add decision-support regimes that translate UVIF scores into actionable operational states.",
    "",
    "New Array-oriented outputs:",
]
array_summary_lines.extend(f"- {path}" for path in array_upgrade_files)
array_summary_lines.extend([
    "",
    "Recommended manuscript placement:",
    "- Methodology: describe ablation variants and bootstrap boundary confidence estimation.",
    "- Results: include fig_array_ablation_boundary_comparison.png and manuscript_table_array_main_findings.csv.",
    "- Discussion: emphasize that UVIF adds operational decision intelligence, not composable security proof replacement.",
    "- Limitations: retain the BB84-inspired and simulation-based scope statement.",
])

array_summary_path = OUT_DIR / "outputs_summary_array_targeted_upgrade.txt"
with open(array_summary_path, "w", encoding="utf-8") as f:
    f.write("\n".join(array_summary_lines))

log(f"[ARRAY-UPGRADE] Summary saved: {array_summary_path}")
print("\n".join(array_summary_lines[:28]))
